## Baseline Models

In [1]:
import sys, numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

sys.path.insert(0, str(Path('..').resolve()))
from src.pipeline import load_and_clean, build_pipeline

In [2]:
df_clean, y = load_and_clean('../data/processed/matches.parquet')

index_trainval, index_test = train_test_split(
    np.arange(len(df_clean)), test_size=0.20, random_state=42, stratify=y)
index_train, index_val = train_test_split(
    index_trainval, test_size=0.125, random_state=42, stratify=y.iloc[index_trainval])

pipeline = build_pipeline()
X_train = pipeline.fit_transform(df_clean.iloc[index_train])
X_val   = pipeline.transform(df_clean.iloc[index_val])
y_train = y.iloc[index_train].values
y_val   = y.iloc[index_val].values

print(f'X_train {X_train.shape}, X_val {X_val.shape}')

X_train (140, 408), X_val (20, 408)


In [3]:
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support

def evaluate_model(model, X, y):
    proba = model.predict_proba(X)[:, 1]
    preds = model.predict(X)
    precision, recall, f1, _ = precision_recall_fscore_support(y, preds, average='macro')
    return {
        'roc_auc': roc_auc_score(y, proba),
        'accuracy': accuracy_score(y, preds),
        'macro_precision': precision,
        'macro_recall': recall,
        'macro_f1': f1,
    }

In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    'DummyClassifier': DummyClassifier(strategy='most_frequent'),
    'LogisticRegression': LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs'),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.3,
                             random_state=42, n_jobs=-1),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    train_metrics = evaluate_model(model, X_train, y_train)
    val_metrics   = evaluate_model(model, X_val, y_val)
    row = {'model': name}
    row.update({f'train_{k}': v for k, v in train_metrics.items()})
    row.update({f'val_{k}': v for k, v in val_metrics.items()})
    results.append(row)

results_df = pd.DataFrame(results).set_index('model')
display(results_df.round(4))

/Users/zhangzhi/miniconda3/envs/dev/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/zhangzhi/miniconda3/envs/dev/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,train_roc_auc,train_accuracy,train_macro_precision,train_macro_recall,train_macro_f1,val_roc_auc,val_accuracy,val_macro_precision,val_macro_recall,val_macro_f1
model,,,,,,,,,,
DummyClassifier,0.5,0.5357,0.2679,0.5,0.3488,0.5000,0.55,0.2750,0.5000,0.3548
LogisticRegression,1.0,1.0000,1.0000,1.0,1.0000,0.9192,0.80,0.7980,0.7980,0.7980
RandomForest,1.0,1.0000,1.0000,1.0,1.0000,1.0000,0.95,0.9583,0.9444,0.9488
XGBoost,1.0,1.0000,1.0000,1.0,1.0000,1.0000,1.00,1.0000,1.0000,1.0000


In [5]:
for _, row in results_df.iterrows():
    gap = row['train_accuracy'] - row['val_accuracy']
    flag = '  ** OVERFIT **' if gap > 0.05 else ''
    print(f'{row.name:20s}  train-val acc gap: {gap:+.4f}{flag}')

leakage = results_df[results_df['val_accuracy'] > 0.72]
if len(leakage):
    print(f'\nLEAKAGE WARNING: {list(leakage.index)} val acc > 72%')
leakage_auc = results_df[results_df['val_roc_auc'] > 0.78]
if len(leakage_auc):
    print(f'LEAKAGE WARNING: {list(leakage_auc.index)} val AUC > 0.78')

DummyClassifier       train-val acc gap: -0.0143
LogisticRegression    train-val acc gap: +0.2000  ** OVERFIT **
RandomForest          train-val acc gap: +0.0500  ** OVERFIT **
XGBoost               train-val acc gap: +0.0000

LEAKAGE WARNING: ['LogisticRegression', 'RandomForest', 'XGBoost'] val acc > 72%
LEAKAGE WARNING: ['LogisticRegression', 'RandomForest', 'XGBoost'] val AUC > 0.78


## LR Hyperparameter Search

In [6]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    LogisticRegression(max_iter=1000, solver='lbfgs'),
    param_grid={'C': [0.01, 0.1, 1, 10, 100]},
    cv=5, scoring='roc_auc', n_jobs=-1,
)
grid_search.fit(X_train, y_train)

print(f'best C={grid_search.best_params_["C"]}, CV AUC={grid_search.best_score_:.4f}')

best_lr = grid_search.best_estimator_
train_metrics = evaluate_model(best_lr, X_train, y_train)
val_metrics   = evaluate_model(best_lr, X_val, y_val)
tuned_row = {'model': f'LR_tuned(C={grid_search.best_params_["C"]})'}
tuned_row.update({f'train_{k}': v for k, v in train_metrics.items()})
tuned_row.update({f'val_{k}': v for k, v in val_metrics.items()})
results.append(tuned_row)

print(f'val AUC: {val_metrics["roc_auc"]:.4f}, val acc: {val_metrics["accuracy"]:.4f}')

best C=0.01, CV AUC=0.9723
val AUC: 0.9394, val acc: 0.8500


In [7]:
full_results = pd.DataFrame(results).set_index('model').sort_values('val_roc_auc', ascending=False)
display(full_results.round(4))

,train_roc_auc,train_accuracy,train_macro_precision,train_macro_recall,train_macro_f1,val_roc_auc,val_accuracy,val_macro_precision,val_macro_recall,val_macro_f1
model,,,,,,,,,,
RandomForest,1.0,1.0000,1.0000,1.0,1.0000,1.0000,0.95,0.9583,0.9444,0.9488
XGBoost,1.0,1.0000,1.0000,1.0,1.0000,1.0000,1.00,1.0000,1.0000,1.0000
LR_tuned(C=0.01),1.0,1.0000,1.0000,1.0,1.0000,0.9394,0.85,0.8542,0.8434,0.8465
LogisticRegression,1.0,1.0000,1.0000,1.0,1.0000,0.9192,0.80,0.7980,0.7980,0.7980
DummyClassifier,0.5,0.5357,0.2679,0.5,0.3488,0.5000,0.55,0.2750,0.5000,0.3548


## Ablation: drop player_hero_wr

In [8]:
hero_wr_cols = [c for c in df_clean.columns if 'player_hero_wr' in c]
print(f'dropping {len(hero_wr_cols)} cols: {hero_wr_cols[:4]} ...')

df_no_hero_wr = df_clean.drop(columns=hero_wr_cols)

pipeline_no_hero_wr = build_pipeline()
X_train_no_wr = pipeline_no_hero_wr.fit_transform(df_no_hero_wr.iloc[index_train])
X_val_no_wr   = pipeline_no_hero_wr.transform(df_no_hero_wr.iloc[index_val])

print(f'X_train_no_wr {X_train_no_wr.shape}, X_val_no_wr {X_val_no_wr.shape}')

dropping 14 cols: ['t0_p0_player_hero_wr', 't0_p1_player_hero_wr', 't0_p2_player_hero_wr', 't0_p3_player_hero_wr'] ...
X_train_no_wr (140, 394), X_val_no_wr (20, 394)


In [9]:
ablation_models = {
    'DummyClassifier': DummyClassifier(strategy='most_frequent'),
    'LogisticRegression': LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs'),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.3,
                             random_state=42, n_jobs=-1),
}

ablation_results = []
for name, model in ablation_models.items():
    model.fit(X_train_no_wr, y_train)
    train_metrics = evaluate_model(model, X_train_no_wr, y_train)
    val_metrics   = evaluate_model(model, X_val_no_wr, y_val)
    row = {'model': name}
    row.update({f'train_{k}': v for k, v in train_metrics.items()})
    row.update({f'val_{k}': v for k, v in val_metrics.items()})
    ablation_results.append(row)

# tuned LR with same best C from full-feature search
best_C = grid_search.best_params_['C']
lr_tuned_no_wr = LogisticRegression(C=best_C, max_iter=1000, solver='lbfgs')
lr_tuned_no_wr.fit(X_train_no_wr, y_train)
train_metrics = evaluate_model(lr_tuned_no_wr, X_train_no_wr, y_train)
val_metrics   = evaluate_model(lr_tuned_no_wr, X_val_no_wr, y_val)
tuned_row = {'model': f'LR_tuned(C={best_C})'}
tuned_row.update({f'train_{k}': v for k, v in train_metrics.items()})
tuned_row.update({f'val_{k}': v for k, v in val_metrics.items()})
ablation_results.append(tuned_row)

ablation_df = pd.DataFrame(ablation_results).set_index('model').sort_values('val_roc_auc', ascending=False)
display(ablation_df.round(4))

/Users/zhangzhi/miniconda3/envs/dev/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/zhangzhi/miniconda3/envs/dev/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,train_roc_auc,train_accuracy,train_macro_precision,train_macro_recall,train_macro_f1,val_roc_auc,val_accuracy,val_macro_precision,val_macro_recall,val_macro_f1
model,,,,,,,,,,
XGBoost,1.0000,1.0000,1.0000,1.0000,1.0000,0.9697,0.85,0.8542,0.8434,0.8465
RandomForest,1.0000,1.0000,1.0000,1.0000,1.0000,0.8990,0.90,0.9231,0.8889,0.8958
LR_tuned(C=0.01),0.9994,0.9786,0.9790,0.9779,0.9784,0.8485,0.80,0.7980,0.7980,0.7980
LogisticRegression,1.0000,1.0000,1.0000,1.0000,1.0000,0.8283,0.80,0.7980,0.7980,0.7980
DummyClassifier,0.5000,0.5357,0.2679,0.5000,0.3488,0.5000,0.55,0.2750,0.5000,0.3548


In [10]:
compare = full_results[['val_roc_auc', 'val_accuracy']].rename(
    columns={'val_roc_auc': 'full_auc', 'val_accuracy': 'full_acc'})
compare = compare.join(
    ablation_df[['val_roc_auc', 'val_accuracy']].rename(
        columns={'val_roc_auc': 'ablated_auc', 'val_accuracy': 'ablated_acc'}))
compare['delta_auc'] = compare['ablated_auc'] - compare['full_auc']
compare['delta_acc'] = compare['ablated_acc'] - compare['full_acc']
display(compare.round(4))

,full_auc,full_acc,ablated_auc,ablated_acc,delta_auc,delta_acc
model,,,,,,
RandomForest,1.0000,0.95,0.8990,0.90,-0.1010,-0.05
XGBoost,1.0000,1.00,0.9697,0.85,-0.0303,-0.15
LR_tuned(C=0.01),0.9394,0.85,0.8485,0.80,-0.0909,-0.05
LogisticRegression,0.9192,0.80,0.8283,0.80,-0.0909,0.00
DummyClassifier,0.5000,0.55,0.5000,0.55,0.0000,0.00


## Test Set Evaluation

In [11]:
X_test = pipeline.transform(df_clean.iloc[index_test])
y_test = y.iloc[index_test].values

X_test_no_wr = pipeline_no_hero_wr.transform(df_no_hero_wr.iloc[index_test])

print(f'X_test {X_test.shape}, X_test_no_wr {X_test_no_wr.shape}')

X_test (40, 408), X_test_no_wr (40, 394)


In [12]:
test_rows = []

for name, model in models.items():
    metrics = evaluate_model(model, X_test, y_test)
    test_rows.append({'model': name, 'setting': 'full', **metrics})

test_rows.append({'model': f'LR_tuned(C={grid_search.best_params_["C"]})', 'setting': 'full',
                  **evaluate_model(best_lr, X_test, y_test)})

for name, model in ablation_models.items():
    metrics = evaluate_model(model, X_test_no_wr, y_test)
    test_rows.append({'model': name, 'setting': 'no_hero_wr', **metrics})

test_rows.append({'model': f'LR_tuned(C={best_C})', 'setting': 'no_hero_wr',
                  **evaluate_model(lr_tuned_no_wr, X_test_no_wr, y_test)})

test_df = pd.DataFrame(test_rows).set_index(['setting', 'model']).sort_index()
display(test_df.round(4))

/Users/zhangzhi/miniconda3/envs/dev/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/zhangzhi/miniconda3/envs/dev/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


roc_auc  accuracy  macro_precision  \
setting    model                                                    
full       DummyClassifier      0.5000     0.525           0.2625   
           LR_tuned(C=0.01)     0.9774     0.875           0.8763   
           LogisticRegression   0.9674     0.925           0.9375   
           RandomForest         0.9812     0.875           0.8854   
           XGBoost              0.9975     0.925           0.9375   
no_hero_wr DummyClassifier      0.5000     0.525           0.2625   
           LR_tuned(C=0.01)     0.9549     0.825           0.8258   
           LogisticRegression   0.9373     0.825           0.8333   
           RandomForest         0.9023     0.800           0.8031   
           XGBoost              0.8922     0.800           0.8031   

                               macro_recall  macro_f1  
setting    model                                       
full       DummyClassifier           0.5000    0.3443  
           LR_tuned(C=0.01)          0.8734    0.8743  
           LogisticRegression        0.9211    0.9238  
           RandomForest              0.8709    0.8730  
           XGBoost                   0.9211    0.9238  
no_hero_wr DummyClassifier           0.5000    0.3443  
           LR_tuned(C=0.01)          0.8233    0.8240  
           LogisticRegression        0.8208    0.8222  
           RandomForest              0.7970    0.7980  
           XGBoost                   0.7970    0.7980